# Winning Without an xG Advantage
## A Statistical Analysis of Positional Structure in xG-Parity Matches

**Georgia Tech MSA Spring 2026**  
**Team 4:** Alexander Avramov, Noah Boonin, Thomas LaRock  
**Track:** Soccer Analytics Dashboard

---

### Executive Summary

*[Write this LAST after completing all analysis - this is a placeholder]*

This notebook presents a systematic, question-driven analysis of 651 professional soccer matches where both teams created nearly equal expected goals (|ΔxG| ≤ 0.3). Through rigorous statistical testing, we demonstrate that **when chance quality is equal, positional structure separates winners from non-winners**.

**Key Findings:**
1. [Finding 1 with specific numbers and p-value]
2. [Finding 2 with specific numbers and p-value]
3. [Finding 3 with specific numbers and p-value]
4. [Finding 4 with specific numbers and p-value]

**Statistical Rigor:**
- 7 distinct statistical tests applied
- Multiple testing correction (Bonferroni)
- Effect size analysis (Cohen's d)
- Multivariate testing (Hotelling's T²)

**Implications:**
[What this means for coaches, analysts, and tactical understanding]

---

### Setup: Library Imports

In [1]:
import pandas as pd
import numpy as np
import polars as pl
from pathlib import Path

from scipy import stats
from scipy.stats import (
    chi2_contingency,
    mannwhitneyu,
    spearmanr,
    f
)
from numpy.linalg import inv

import matplotlib.pyplot as plt
import seaborn as sns
from mplsoccer import Pitch, VerticalPitch

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
plt.style.use('seaborn-v0_8-darkgrid')

print("Libraries imported successfully")

Libraries imported successfully


### Data Loading

We use **StatsBomb Open Data** containing event-level data from professional soccer matches across multiple competitions and seasons.

**Data Structure:**
- `matches.parquet` - Match-level metadata (teams, scores, competitions)
- `events.parquet` - Event-level data (passes, shots, carries, etc.)
- `lineups.parquet` - Player lineups and positions

In [3]:
# Data directory
DATA_DIR = Path("../data/raw")
STATSBOMB_DIR = DATA_DIR / "Statsbomb"

# Load core datasets
print("Loading StatsBomb data...")
matches = pl.read_parquet(STATSBOMB_DIR / "matches.parquet")
events = pl.read_parquet(STATSBOMB_DIR / "events.parquet")
lineups = pl.read_parquet(STATSBOMB_DIR / "lineups.parquet")

print(f"Loaded {len(matches):,} matches")
print(f"Loaded {len(events):,} events")
print(f"Loaded {len(lineups):,} lineup records")
print(f"\nDataset date range: {matches['match_date'].min()} to {matches['match_date'].max()}")
print(f"Competitions: {matches['competition'].n_unique()} unique competitions")

Loading StatsBomb data...
Loaded 3,464 matches
Loaded 12,188,949 events
Loaded 165,820 lineup records

Dataset date range: 1958-06-24 to 2025-07-27
Competitions: 21 unique competitions


### Data Quality Verification

The StatsBomb event dataset is highly sparse by design, as most columns are event-conditional.
Fields related to shots, goalkeeper actions, substitutions, disciplinary events, and relational links
(e.g., assisted shots) are only populated when the corresponding event type occurs.

As a result, columns such as `shot_statsbomb_xg`, `goalkeeper_*`, and `substitution_*` exhibit high
missingness due to event rarity rather than data quality issues.

Core identifiers, timestamps, team labels, and possession fields show no missingness, confirming the
structural integrity of the dataset. Spatial fields and player metadata are nearly complete, with
minor missingness attributable to off-camera or administrative events.

No global imputation or row-level filtering is required at this stage; subsequent analyses will
filter by event type and operate only on contextually relevant fields.

In [5]:
# Check for missing values in critical fields
print("Data Completeness Check:")
print(f"{'='*60}")

# Matches
missing_matches = matches.select([
    pl.col("match_id").is_null().sum().alias("match_id"),
    pl.col("home_score").is_null().sum().alias("home_score"),
    pl.col("away_score").is_null().sum().alias("away_score"),
])
print(f"Matches - Missing values: {missing_matches}")

# Events
missing_events = events.select([
    pl.col("match_id").is_null().sum().alias("match_id"),
    pl.col("type").is_null().sum().alias("type"),
    pl.col("team").is_null().sum().alias("team"),
])
print(f"Events - Missing values: {missing_events}")

# Check for duplicate match IDs
duplicate_matches = matches.group_by("match_id").agg(pl.len()).filter(pl.col("len") > 1)
print(f"\nDuplicate matches: {len(duplicate_matches)}")

print(f"{'='*60}")
print("Data quality check complete")

Data Completeness Check:
Matches - Missing values: shape: (1, 3)
┌──────────┬────────────┬────────────┐
│ match_id ┆ home_score ┆ away_score │
│ ---      ┆ ---        ┆ ---        │
│ u32      ┆ u32        ┆ u32        │
╞══════════╪════════════╪════════════╡
│ 0        ┆ 0          ┆ 0          │
└──────────┴────────────┴────────────┘
Events - Missing values: shape: (1, 3)
┌──────────┬──────┬──────┐
│ match_id ┆ type ┆ team │
│ ---      ┆ ---  ┆ ---  │
│ u32      ┆ u32  ┆ u32  │
╞══════════╪══════╪══════╡
│ 0        ┆ 0    ┆ 0    │
└──────────┴──────┴──────┘

Duplicate matches: 0
Data quality check complete


# SECTION 1: Validating the xG-Parity Concept

## Question 1: Are xG-parity matches actually "close" matches?

**Research Question:** If xG-parity (|ΔxG| ≤ 0.3) truly represents "close" matches, we should observe:
1. Higher draw rate than in xG-dominant matches
2. More balanced win/loss distribution

**Hypothesis:** Draw rate in xG-parity matches > Draw rate in all matches

**xG Difference Categories:** Parity (≤0.3) | Close (0.3–0.7) | Moderate (0.7–1.5) | Dominant (>1.5)
**Statistical Test:** Chi-square test of independence  
**Null Hypothesis (H₀):** Outcome distribution is independent of xG difference category  
**Alternative Hypothesis (H₁):** xG-parity matches have different outcome distribution

**Expected Result:** χ² p-value < 0.05, with higher draw rate in parity matches

---

In [6]:
# Build match-level xG and outcome data
shots = events.filter(pl.col("type") == "Shot")

team_match_xg = (
    shots
    .group_by(["match_id", "team"])
    .agg(pl.col("shot_statsbomb_xg").sum().alias("total_xg"))
)

# Build team outcomes from match scores
home_outcomes = matches.select([
    pl.col("match_id"),
    pl.col("home_team").alias("team"),
    pl.col("home_score"),
    pl.col("away_score"),
]).with_columns([
    pl.when(pl.col("home_score") > pl.col("away_score")).then(pl.lit("Win"))
      .when(pl.col("home_score") < pl.col("away_score")).then(pl.lit("Loss"))
      .otherwise(pl.lit("Draw")).alias("outcome")
])

away_outcomes = matches.select([
    pl.col("match_id"),
    pl.col("away_team").alias("team"),
    pl.col("home_score"),
    pl.col("away_score"),
]).with_columns([
    pl.when(pl.col("away_score") > pl.col("home_score")).then(pl.lit("Win"))
      .when(pl.col("away_score") < pl.col("home_score")).then(pl.lit("Loss"))
      .otherwise(pl.lit("Draw")).alias("outcome")
])

team_outcomes = pl.concat([
    home_outcomes.select(["match_id", "team", "outcome"]),
    away_outcomes.select(["match_id", "team", "outcome"])
])

team_match_xg = team_match_xg.join(team_outcomes, on=["match_id", "team"], how="left")

# Pivot to one row per match with both teams' xG
xg_wide = (
    team_match_xg
    .join(
        team_match_xg.rename({"team": "opp_team", "total_xg": "opp_xg", "outcome": "opp_outcome"}),
        on="match_id"
    )
    .filter(pl.col("team") != pl.col("opp_team"))
    # Deduplicate to one row per match (home team perspective)
    .join(matches.select(["match_id", pl.col("home_team").alias("team")]), on=["match_id", "team"])
)

xg_wide = xg_wide.with_columns([
    (pl.col("total_xg") - pl.col("opp_xg")).abs().alias("xg_diff")
])

# Assign xG difference buckets
xg_wide = xg_wide.with_columns([
    pl.when(pl.col("xg_diff") <= 0.3).then(pl.lit("Parity (≤0.3)"))
      .when(pl.col("xg_diff") <= 0.7).then(pl.lit("Close (0.3–0.7)"))
      .when(pl.col("xg_diff") <= 1.5).then(pl.lit("Moderate (0.7–1.5)"))
      .otherwise(pl.lit("Dominant (>1.5)")).alias("xg_category")
])

print(f"Match-level observations: {len(xg_wide):,}")
print(f"\nMatches per xG category:")
print(xg_wide.group_by("xg_category").agg(pl.len().alias("count")).sort("count", descending=True))

Match-level observations: 3,457

Matches per xG category:
shape: (4, 2)
┌────────────────────┬───────┐
│ xg_category        ┆ count │
│ ---                ┆ ---   │
│ str                ┆ u32   │
╞════════════════════╪═══════╡
│ Moderate (0.7–1.5) ┆ 1127  │
│ Dominant (>1.5)    ┆ 870   │
│ Close (0.3–0.7)    ┆ 809   │
│ Parity (≤0.3)      ┆ 651   │
└────────────────────┴───────┘


In [7]:
from scipy.stats import chi2_contingency

# Convert to pandas for scipy
df = xg_wide.select(["match_id", "xg_category", "outcome"]).to_pandas()

# Build contingency table: xG category × outcome
contingency = pd.crosstab(df["xg_category"], df["outcome"])

# Define category order for display
cat_order = ["Parity (≤0.3)", "Close (0.3–0.7)", "Moderate (0.7–1.5)", "Dominant (>1.5)"]
contingency = contingency.reindex(cat_order)

# Run chi-square test
chi2, p, dof, expected = chi2_contingency(contingency)

# Add draw rate column for interpretability
contingency["Total"] = contingency.sum(axis=1)
contingency["Draw Rate"] = (contingency["Draw"] / contingency["Total"]).round(3)

print("Contingency Table: Match Outcome by xG Difference Category")
print("=" * 65)
print(contingency.to_string())
print(f"\nChi-Square Test of Independence")
print(f"  χ² statistic : {chi2:.4f}")
print(f"  Degrees of freedom: {dof}")
print(f"  p-value      : {p:.6f}")
print(f"\nInterpretation: {'Reject H₀' if p < 0.05 else 'Fail to reject H₀'} at α = 0.05")

Contingency Table: Match Outcome by xG Difference Category
outcome             Draw  Loss  Win  Total  Draw Rate
xg_category                                          
Parity (≤0.3)        194   204  253    651      0.298
Close (0.3–0.7)      240   263  306    809      0.297
Moderate (0.7–1.5)   266   356  505   1127      0.236
Dominant (>1.5)       97   277  496    870      0.111

Chi-Square Test of Independence
  χ² statistic : 124.4842
  Degrees of freedom: 6
  p-value      : 0.000000

Interpretation: Reject H₀ at α = 0.05


### Finding 1: xG-Parity Matches Are Genuinely Contested

**Statistical Evidence:**
- Test: Chi-square test of independence
- χ² = 124.48, df = 6, p < 0.0001
- Result: Reject H₀ — outcome distribution is not independent of xG category

**Draw Rates by Category:**
| xG Category | Draw Rate |
|-------------|-----------|
| Parity (≤0.3) | 29.8% |
| Close (0.3–0.7) | 29.7% |
| Moderate (0.7–1.5) | 23.6% |
| Dominant (>1.5) | 11.1% |

**Interpretation:**  
Parity and close matches share nearly identical draw rates (~30%), both roughly 3× higher 
than dominant matches (11.1%). The meaningful threshold is not precisely at 0.3 — it falls 
somewhere between 0.7 and 1.5, where one team's xG advantage begins translating reliably 
into wins. Within the parity subset (|ΔxG| ≤ 0.3), outcomes remain genuinely uncertain: 
draws occur in nearly 1 in 3 matches, and no single outcome dominates.

**Decision:** xG-parity matches are validated as genuinely contested 

---

---

# Section 2: Positional Touch Share Analysis

## Question 2: When xG is equal, do winners possess the ball differently across positions?

**Hypothesis:** Winning teams shift possession forward — allocating more touches to attacking 
positions (AM, WF, ST) and fewer to defensive positions (CB, DM) — compared to non-winning teams 
in the same xG-parity matches.

**Approach:** We test this in three layers:
- **2A — Is there a difference?** Mann-Whitney U test by position
- **2B — How big is it?** Cohen's d effect sizes
- **2C — Is it robust?** Bonferroni correction for multiple testing

**Data:** 651 xG-parity matches → 1,302 team-match observations  
**Outcome Groups:** Win vs. Non-Win (Draw + Loss combined)

---

In [17]:
# Define touch event types
TOUCH_EVENTS = ["Ball Receipt*", "Pass", "Carry", "Dribble", "Shot"]

# Position bin mapping
POSITION_BIN_MAP = {
    # Goalkeeper
    "Goalkeeper": "Goalkeeper",
    # Center Backs
    "Center Back": "Center Back",
    "Left Center Back": "Center Back",
    "Right Center Back": "Center Back",
    # Fullbacks
    "Left Back": "Fullback",
    "Right Back": "Fullback",
    "Left Wing Back": "Fullback",
    "Right Wing Back": "Fullback",
    # Defensive Midfield
    "Center Defensive Midfield": "Defensive Midfield",
    "Left Defensive Midfield": "Defensive Midfield",
    "Right Defensive Midfield": "Defensive Midfield",
    # Central Midfield
    "Center Midfield": "Central Midfield",
    "Left Center Midfield": "Central Midfield",
    "Right Center Midfield": "Central Midfield",
    # Attacking Midfield
    "Center Attacking Midfield": "Attacking Midfield",
    "Left Attacking Midfield": "Attacking Midfield",
    "Right Attacking Midfield": "Attacking Midfield",
    # Wide Midfield
    "Left Midfield": "Wide Midfield",
    "Right Midfield": "Wide Midfield",
    # Wide Forward
    "Left Wing": "Wide Forward",
    "Right Wing": "Wide Forward",
    # Striker
    "Center Forward": "Striker",
    "Left Center Forward": "Striker",
    "Right Center Forward": "Striker",
    "Secondary Striker": "Striker",
}

# Filter to xG-parity match IDs
parity_match_ids = (
    xg_wide
    .filter(pl.col("xg_category") == "Parity (≤0.3)")
    .select("match_id")
    .to_series()
    .to_list()
)
print(f"Parity match IDs: {len(parity_match_ids):,}")

# Filter events to touch events in parity matches
touches = (
    events
    .filter(pl.col("match_id").is_in(parity_match_ids))
    .filter(pl.col("type").is_in(TOUCH_EVENTS))
    .filter(pl.col("position").is_not_null())
    .with_columns(
        pl.col("position").replace(POSITION_BIN_MAP).alias("position_bin")
    )
    .filter(pl.col("position_bin").is_not_null())
)

# Aggregate touches by team-match-position
team_match_position_touches = (
    touches
    .group_by(["match_id", "team", "position_bin"])
    .agg(pl.len().alias("touches"))
)

# Compute touch share within each team-match
team_match_totals = (
    team_match_position_touches
    .group_by(["match_id", "team"])
    .agg(pl.col("touches").sum().alias("team_touches"))
)

team_match_position_touches = (
    team_match_position_touches
    .join(team_match_totals, on=["match_id", "team"])
    .with_columns(
        (pl.col("touches") / pl.col("team_touches")).alias("touch_share")
    )
    .join(team_outcomes, on=["match_id", "team"])
    .with_columns(
        pl.when(pl.col("outcome") == "Win")
          .then(pl.lit("Win"))
          .otherwise(pl.lit("Non-Win"))
          .alias("outcome_binary")
    )
)

# Convert to pandas for statistical testing
touches_pd = team_match_position_touches.to_pandas()

print(f"Team-match-position observations: {len(touches_pd):,}")
print(f"Unique positions: {sorted(touches_pd['position_bin'].unique())}")
print(f"\nOutcome distribution:")
print(touches_pd.drop_duplicates(['match_id','team'])['outcome_binary'].value_counts())

Parity match IDs: 651
Team-match-position observations: 9,207
Unique positions: ['Attacking Midfield', 'Center Back', 'Central Midfield', 'Defensive Midfield', 'Fullback', 'Goalkeeper', 'Striker', 'Wide Forward', 'Wide Midfield']

Outcome distribution:
outcome_binary
Non-Win    845
Win        457
Name: count, dtype: int64


In [18]:
from scipy.stats import mannwhitneyu

positions = sorted(touches_pd['position_bin'].unique())

wins = touches_pd[touches_pd['outcome_binary'] == 'Win']
nonwins = touches_pd[touches_pd['outcome_binary'] == 'Non-Win']

results = []
for position in positions:
    win_shares = wins[wins['position_bin'] == position]['touch_share']
    nonwin_shares = nonwins[nonwins['position_bin'] == position]['touch_share']
    
    u_stat, p_val = mannwhitneyu(win_shares, nonwin_shares, alternative='two-sided')
    
    win_mean = win_shares.mean()
    nonwin_mean = nonwin_shares.mean()
    diff = win_mean - nonwin_mean
    
    results.append({
        'Position': position,
        'Win Mean': round(win_mean, 4),
        'Non-Win Mean': round(nonwin_mean, 4),
        'Difference': round(diff, 4),
        'U Statistic': round(u_stat, 1),
        'p-value': round(p_val, 4),
        'Significant': '✅' if p_val < 0.05 else '❌'
    })

mw_results = pd.DataFrame(results).sort_values('Difference', ascending=False)
print("Mann-Whitney U Test: Touch Share by Position (Win vs. Non-Win)")
print("=" * 70)
print(mw_results.to_string(index=False))
print(f"\nSignificant positions (p < 0.05): {mw_results['Significant'].eq('✅').sum()}/9")

Mann-Whitney U Test: Touch Share by Position (Win vs. Non-Win)
          Position  Win Mean  Non-Win Mean  Difference  U Statistic  p-value Significant
Attacking Midfield    0.1058        0.0903      0.0155      65242.5   0.0002           ✅
     Wide Midfield    0.1654        0.1528      0.0126      33353.0   0.0429           ✅
      Wide Forward    0.1741        0.1617      0.0124      85302.0   0.0009           ✅
           Striker    0.1171        0.1119      0.0052     201868.5   0.1749           ❌
        Goalkeeper    0.0497        0.0460      0.0038     219014.5   0.0001           ✅
  Central Midfield    0.1782        0.1747      0.0036      68170.0   0.2220           ❌
          Fullback    0.1881        0.1937     -0.0055     171911.0   0.0019           ✅
       Center Back    0.1713        0.1804     -0.0091     174002.0   0.0032           ✅
Defensive Midfield    0.1559        0.1666     -0.0107     149582.0   0.0051           ✅

Significant positions (p < 0.05): 7/9


In [19]:
def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(ddof=1), group2.var(ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return (group1.mean() - group2.mean()) / pooled_std

effect_sizes = []
for position in positions:
    win_shares = wins[wins['position_bin'] == position]['touch_share']
    nonwin_shares = nonwins[nonwins['position_bin'] == position]['touch_share']
    
    d = cohens_d(win_shares, nonwin_shares)
    
    if abs(d) >= 0.8:
        magnitude = 'Large'
    elif abs(d) >= 0.5:
        magnitude = 'Medium'
    elif abs(d) >= 0.2:
        magnitude = 'Small'
    else:
        magnitude = 'Negligible'
    
    effect_sizes.append({
        'Position': position,
        'Cohen\'s d': round(d, 4),
        'Magnitude': magnitude,
    })

cd_results = pd.DataFrame(effect_sizes).sort_values('Cohen\'s d', ascending=False)
print("Cohen's d Effect Sizes: Touch Share by Position (Win vs. Non-Win)")
print("=" * 55)
print(cd_results.to_string(index=False))

Cohen's d Effect Sizes: Touch Share by Position (Win vs. Non-Win)
          Position  Cohen's d  Magnitude
Attacking Midfield     0.3527      Small
      Wide Forward     0.2106      Small
        Goalkeeper     0.2028      Small
     Wide Midfield     0.1688 Negligible
           Striker     0.1018 Negligible
  Central Midfield     0.0508 Negligible
       Center Back    -0.1456 Negligible
Defensive Midfield    -0.1767 Negligible
          Fullback    -0.1866 Negligible


In [20]:
ALPHA = 0.05
n_tests = len(positions)
bonferroni_alpha = ALPHA / n_tests

bonferroni_results = mw_results.copy()
bonferroni_results['Survives Bonferroni'] = bonferroni_results['p-value'].apply(
    lambda p: '✅' if p < bonferroni_alpha else '❌'
)

print(f"Bonferroni Correction: α = {ALPHA} / {n_tests} tests = {bonferroni_alpha:.4f}")
print("=" * 75)
print(bonferroni_results[['Position', 'p-value', 'Significant', 'Survives Bonferroni']].to_string(index=False))
print(f"\nPositions surviving Bonferroni: {bonferroni_results['Survives Bonferroni'].eq('✅').sum()}/{n_tests}")

Bonferroni Correction: α = 0.05 / 9 tests = 0.0056
          Position  p-value Significant Survives Bonferroni
Attacking Midfield   0.0002           ✅                   ✅
     Wide Midfield   0.0429           ✅                   ❌
      Wide Forward   0.0009           ✅                   ✅
           Striker   0.1749           ❌                   ❌
        Goalkeeper   0.0001           ✅                   ✅
  Central Midfield   0.2220           ❌                   ❌
          Fullback   0.0019           ✅                   ✅
       Center Back   0.0032           ✅                   ✅
Defensive Midfield   0.0051           ✅                   ✅

Positions surviving Bonferroni: 6/9


### Finding 2: Positional Structure Systematically Differs Between Winners and Non-Winners

**Statistical Evidence Summary:**

| Position | Win Mean | Non-Win Mean | Difference | p-value | Cohen's d | Survives Bonferroni |
|----------|----------|--------------|------------|---------|-----------|---------------------|
| Attacking Midfield | 10.6% | 9.0% | +1.6pp | 0.0002 | 0.35 | ✅ |
| Wide Forward | 17.4% | 16.2% | +1.2pp | 0.0009 | 0.21 | ✅ |
| Goalkeeper | 5.0% | 4.6% | +0.4pp | 0.0001 | 0.20 | ✅ |
| Wide Midfield | 16.5% | 15.3% | +1.3pp | 0.0429 | 0.17 | ❌ |
| Striker | 11.7% | 11.2% | +0.5pp | 0.1749 | 0.10 | ❌ |
| Central Midfield | 17.8% | 17.5% | +0.4pp | 0.2220 | 0.05 | ❌ |
| Fullback | 18.8% | 19.4% | -0.6pp | 0.0019 | -0.19 | ✅ |
| Center Back | 17.1% | 18.0% | -0.9pp | 0.0032 | -0.15 | ✅ |
| Defensive Midfield | 15.6% | 16.7% | -1.1pp | 0.0051 | -0.18 | ✅ |

**Interpretation:**  
6 of 9 positions show statistically robust differences that survive Bonferroni correction. 
The pattern is consistent and directional: winning teams allocate more touches to attacking 
positions (Attacking Midfield, Wide Forward) and fewer to defensive positions (Center Back, 
Defensive Midfield, Fullback). Effect sizes are small but meaningful in a tactical context — 
soccer outcomes are rarely explained by a single dominant factor.

The Goalkeeper finding is notable: winners build more from the back, suggesting a deliberate 
possession-based style rather than direct play.

**Three-Test Summary:**
- **Mann-Whitney U:** 7/9 positions significant (p < 0.05)
- **Cohen's d:** Effects range from negligible to small (d = 0.05–0.35)
- **Bonferroni:** 6/9 positions robust after multiple testing correction

**Decision:** Positional structure reliably separates winners from non-winners in xG-parity matches 

---